## Multi-Query / Self-Query Retriever

벡터 검색의 한계를 보완하는 두 가지 LangChain Retriever를 다룬다.

| 구분 | Multi-Query | Self-Query |
|---|---|---|
| 핵심 | 질문을 **여러 관점으로 재작성** 후 검색·합집합 | 자연어에서 **검색어 + 메타데이터 필터**를 분리 |
| 목적 | 재현율(recall) 향상 | 정밀도(precision) / 조건부 검색 |
| 적합한 경우 | 표현이 다양하거나 모호한 질문 | year, category 등 구조화 메타데이터가 있는 코퍼스 |

LangChain v1에서는 두 Retriever 모두 `langchain-classic`에 있다.

```text
pip install langchain-classic langchain-community langchain-openai chromadb faiss-cpu
```

#### 기술 배경: 왜 쿼리 변환이 필요한가

밀집 검색(dense retrieval)은 질문과 문서를 같은 임베딩 공간에 투영한 뒤 코사인 유사도·내적으로 순위를 매긴다.  
이 방식은 동의어·문맥에는 강하지만 아래 한계가 있다.

- **표현 민감성**: 질문이 구어체이고 문서가 논문체이면, 의미가 같아도 벡터가 멀어질 수 있다.
- **단일 관점**: 한 번의 질의 임베딩은 하나의 의미 축만 포착한다. 다의적·복합 질문은 관련 문서를 놓치기 쉽다.
- **구조화 조건 무시**: "2024년 자료만" 같은 메타데이터 조건은 임베딩만으로는 안정적으로 강제하기 어렵다.

Multi-Query는 전자(표현·관점)를, Self-Query는 후자(메타데이터 조건)를 보완하는 쿼리 변환(query transformation) 기법이다.  
HyDE(가상 문서 임베딩)와 함께 쓰면 "표현 간극 + 재현율 + 조건부 필터"를 한 파이프라인에서 라우팅할 수 있다.

In [1]:
import os
from dotenv import load_dotenv

# .env 파일의 내용 불러오기
# OPENAI_API_KEY 등 LLM·임베딩 호출에 필요한 환경변수를 로드한다.
# 경로가 다르면 본인 환경에 맞게 수정한다.
load_dotenv("C:/env/.env")

True

### [0] 공통 준비: LLM, 임베딩, 헬퍼

#### 기술 문서: RAG 공통 구성요소

| 구성요소 | 역할 | 본 노트북 선택 |
|---|---|---|
| **LLM** | 질의 재작성·구조화 쿼리 생성·최종 답변 | `gpt-4o-mini`, `temperature=0` (결정적 출력) |
| **Embeddings** | 텍스트 → 벡터 | `text-embedding-3-small` (비용·성능 균형) |
| **Document** | `page_content` + `metadata` | 검색 본문과 필터용 속성을 분리 |
| **LCEL 체인** | `prompt \| llm \| parser` | 파이프라인을 조합형으로 구성 |

`temperature=0`을 쓰는 이유: Multi-Query·Self-Query 모두 "같은 질문에 대해 재현 가능한 변환"이 학습·디버깅에 유리하다.  
창의적 재작성보다 **의도를 유지한 패러프레이즈·필터 추출**이 목표이므로 낮은 temperature가 적합하다.

In [2]:
from typing import List

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 질의 재작성·필터 추출·답변 생성에 공통으로 사용하는 채팅 모델.
# temperature=0: 출력 분산을 줄여 디버깅·비교 실험이 쉬워진다.
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 문서 인덱싱·질의 임베딩에 사용하는 임베딩 모델.
# 인덱스와 질의는 반드시 같은 임베딩 모델을 써야 벡터 공간이 일치한다.
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")


def format_docs(documents: List[Document]) -> str:
    """검색된 Document 목록을 LLM 문맥용 문자열로 변환한다.

    metadata를 함께 출력하면 Self-Query 필터가 실제로 반영됐는지
    (year, category 등) 눈으로 검증하기 쉽다.
    """
    lines = []
    for i, d in enumerate(documents):
        # metadata dict → "k=v, k=v" 형태의 한 줄 요약
        meta = ", ".join(f"{k}={v}" for k, v in d.metadata.items())
        lines.append(f"[{i+1}] ({meta}) {d.page_content}")
    return "\n\n".join(lines)


# grounded generation: 검색 문맥 밖의 지식을 쓰지 않도록 시스템 프롬프트로 제한한다.
answer_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "아래 문맥만 근거로 질문에 답하라. 문맥에 없으면 모른다고 말하라.\n\n[문맥]\n{context}",
        ),
        ("human", "{question}"),
    ]
)
# LCEL: Prompt → LLM → 문자열 파서. Retriever와 독립적으로 재사용한다.
answer_chain = answer_prompt | llm | StrOutputParser()

print("LLM / Embeddings / 헬퍼 준비 완료")

c:\Users\storm\AppData\Local\Programs\Python\Python311\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.3.0) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


LLM / Embeddings / 헬퍼 준비 완료


---
## Part A. Multi-Query Retriever

거리 기반 벡터 검색은 질문 표현이 조금만 달라도 결과가 달라질 수 있다.  
`MultiQueryRetriever`는 LLM으로 **같은 의도·다른 표현**의 질의 여러 개를 만들고, 각 질의로 검색한 뒤 **중복을 제거한 합집합**을 반환한다.

### 흐름
1. 사용자 질문 입력
2. LLM이 대체 질의 3개(기본) 생성
3. 각 질의로 base retriever 검색
4. 결과 합집합(unique union)
5. (선택) 검색 문서로 최종 답변 생성

#### 기술 문서: Multi-Query의 동작 원리

**문제 정의**  
ANN(Approximate Nearest Neighbor) 검색은 질의 벡터 주변의 이웃만 반환한다.  
질의가 문서 표현과 어긋나면 정답 문서가 neighborhood 밖에 있을 수 있다.

**해결 아이디어**  
하나의 의도 $q$를 여러 표현 $\{q_1, q_2, \dots, q_n\}$으로 펼친다.  
각 $q_i$로 top-$k$를 검색한 뒤 합집합을 취하면, 서로 다른 의미 축에서 끌어온 후보가 모여 **재현율(recall)** 이 올라간다.

**비용 모델**
- LLM 호출: 질의 생성 1회 (기본 3개 패러프레이즈)
- 검색 호출: $n$회 (또는 `include_original=True`면 $n+1$회)
- 지연·토큰 비용은 대략 선형으로 증가한다. 프로덕션에서는 $n$·base `k`를 함께 튜닝한다.

**병합 방식**  
LangChain 기본 구현은 문서 객체 기준 unique union이다.  
순위 융합(RRF)까지 넣으면 여러 질의에서 동시에 나온 문서에 더 높은 가중치를 줄 수 있으나, 기본 API에는 포함되지 않는다.

**HyDE와의 차이**
| | Multi-Query | HyDE |
|---|---|---|
| 변환 대상 | 질문 → 여러 질문 | 질문 → 가상 답변 문서 |
| 검색 벡터 | 각 대체 질문의 임베딩 | 가상 문서의 임베딩 |
| 강점 | 관점 다양화·재현율 | 질문-문서 문체 정렬 |

### [A-1] 샘플 코퍼스 & 벡터스토어 (FAISS)

#### 기술 문서: FAISS와 Retriever 추상화

**FAISS (Facebook AI Similarity Search)**  
고차원 벡터의 최근접 이웃 검색 라이브러리다. 로컬·인메모리로 빠르게 실험하기 좋다.  
본 예제는 문서 수가 적어 Flat(정확한) 검색으로 충분하다. 대규모에서는 IVF·HNSW 등 ANN 인덱스를 쓴다.

**`as_retriever(search_kwargs={"k": 2})`**  
VectorStore를 LangChain `BaseRetriever` 인터페이스로 감싼다.  
`MultiQueryRetriever`는 내부에서 base retriever의 `invoke`를 여러 번 호출하므로,  
여기서 `k`는 **대체 질의 하나당** 가져오는 문서 수다.

- base `k=2`, 대체 질의 3개 + 원본 1개 → 최대 $4 \times 2 = 8$개 후보(중복 제거 전)
- 코퍼스가 작으면 unique 결과가 2~3개로 수렴하는 것이 정상이다.

**메타데이터 `topic`**  
Multi-Query 자체는 메타데이터 필터를 쓰지 않는다.  
`topic`은 검색 결과를 사람이 비교·집계하기 위한 라벨이다.

In [3]:
from langchain_community.vectorstores import FAISS

# Multi-Query 효과를 보기 위한 소규모 RAG 기술 노트 코퍼스.
# page_content는 검색 대상 본문, metadata.topic은 결과 비교용 라벨이다.
mq_docs = [
    Document(
        page_content=(
            "하이브리드 검색은 BM25 같은 키워드 검색과 벡터 검색을 결합한다. "
            "정확한 용어 매칭과 의미적 유사성을 동시에 활용할 수 있다. "
            "점수 정규화 후 가중 합산하거나 Reciprocal Rank Fusion(RRF)으로 융합한다."
        ),
        metadata={"topic": "hybrid_search"},
    ),
    Document(
        page_content=(
            "리랭커는 1차 검색 후보를 교차 인코더로 재점수화해 순서를 바꾼다. "
            "재현율을 유지하면서 상위 결과의 정밀도를 높이는 데 효과적이다."
        ),
        metadata={"topic": "reranker"},
    ),
    Document(
        page_content=(
            "쿼리 확장은 사용자 질문을 여러 형태로 바꿔 검색 범위를 넓힌다. "
            "동의어·관련어 추가, 질문 재작성, Multi-Query, HyDE 등이 포함된다."
        ),
        metadata={"topic": "query_expansion"},
    ),
    Document(
        page_content=(
            "청킹은 긴 문서를 검색·임베딩에 맞는 크기로 나눈다. "
            "너무 작으면 문맥이 끊기고, 너무 크면 노이즈가 섞여 품질이 떨어진다."
        ),
        metadata={"topic": "chunking"},
    ),
    Document(
        page_content=(
            "임베딩 모델은 텍스트를 고차원 벡터로 변환한다. "
            "도메인 특화 문서에서는 도메인 적응형 임베딩이 검색 품질을 개선할 수 있다."
        ),
        metadata={"topic": "embedding"},
    ),
    Document(
        page_content=(
            "RAGAS는 faithfulness, answer relevancy, context precision/recall 등으로 "
            "RAG 파이프라인의 검색·생성 품질을 정량 평가한다."
        ),
        metadata={"topic": "evaluation"},
    ),
]

# 문서 본문을 embeddings로 벡터화한 뒤 FAISS 인덱스를 구축한다.
mq_vectorstore = FAISS.from_documents(mq_docs, embeddings)

# MultiQueryRetriever가 감쌀 base retriever.
# k=2: 대체 질의 "하나당" 상위 2개만 가져온다 (전체 k가 아님).
base_retriever = mq_vectorstore.as_retriever(search_kwargs={"k": 2})

print(f"Multi-Query용 문서 수: {len(mq_docs)}")
print(f"base retriever k=2")

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_9768\1803449559.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Multi-Query용 문서 수: 6
base retriever k=2


### [A-2] 베이스라인: 단일 질의 검색

질문을 그대로 한 번만 검색한다. 표현이 문서와 다르면 관련 문서를 놓칠 수 있다.

#### 기술 문서: Naive RAG 검색의 한계

Naive(단일 질의) 검색 파이프라인:

```text
질문 → embed(질문) → ANN top-k → (선택) LLM 답변
```

**실패 패턴**
- 구어체 질문 vs 전문 용어 문서: 임베딩 거리가 커져 오답이 상위권에 들어온다.
- 다의어·은유: 한 벡터가 한 해석만 대표한다.
- 짧은 질문: 정보량이 부족해 neighborhood가 불안정하다.

아래 셀의 질문("말투가 다를 때 검색을 넓혀")은 정답이 `query_expansion`에 가깝다.  
Naive가 이미 맞히더라도, Multi-Query는 **다른 표현에서도 같은 문서를 반복 소환**해 안정성을 높이는 역할이다.

In [4]:
# 의도적으로 구어체·짧은 표현을 사용해 문서 문체와 차이를 둔다.
mq_query = "질문이랑 문서 말투가 다를 때 검색을 어떻게 넓혀?"

# base retriever 한 번 호출 = 질의 1개 × top-k(2)
naive_docs = base_retriever.invoke(mq_query)
print("=== [Naive] 검색 결과 ===")
print(format_docs(naive_docs))
print("\n=== [Naive] 답변 ===")
# 검색 문맥만으로 답변 생성 (grounded generation)
print(
    answer_chain.invoke(
        {"context": format_docs(naive_docs), "question": mq_query}
    )
)

=== [Naive] 검색 결과 ===
[1] (topic=chunking) 청킹은 긴 문서를 검색·임베딩에 맞는 크기로 나눈다. 너무 작으면 문맥이 끊기고, 너무 크면 노이즈가 섞여 품질이 떨어진다.

[2] (topic=query_expansion) 쿼리 확장은 사용자 질문을 여러 형태로 바꿔 검색 범위를 넓힌다. 동의어·관련어 추가, 질문 재작성, Multi-Query, HyDE 등이 포함된다.

=== [Naive] 답변 ===
문맥에 따르면, 질문이랑 문서 말투가 다를 때는 쿼리 확장을 통해 검색 범위를 넓힐 수 있습니다. 쿼리 확장은 사용자 질문을 여러 형태로 바꿔 동의어, 관련어 추가, 질문 재작성 등을 통해 검색 결과를 개선하는 방법입니다.


### [A-3] LangChain `MultiQueryRetriever`

```python
from langchain_classic.retrievers.multi_query import MultiQueryRetriever
```

- `from_llm(retriever, llm)`: 기본 프롬프트로 대체 질의 3개 생성
- `include_original=True`: 원본 질문도 검색에 포함
- `llm_chain.invoke({"question": ...})`으로 생성된 질의 목록을 확인할 수 있다

#### 기술 문서: MultiQueryRetriever 내부 구조

```text
사용자 질문
    │
    ▼
llm_chain  (PromptTemplate | LLM | LineListOutputParser)
    │  줄바꿈으로 구분된 대체 질의 리스트
    ▼
[선택] include_original → 리스트에 원본 추가
    │
    ▼
각 질의마다 base_retriever.invoke(q_i)
    │
    ▼
unique_union(문서들)  → 최종 Document 리스트
```

**`LineListOutputParser`**  
LLM 출력을 `\n`으로 split하고 빈 줄을 제거한다.  
따라서 커스텀 프롬프트도 **한 줄에 질의 하나** 형식을 지켜야 한다. 번호·불릿을 넣으면 파싱 품질이 떨어진다.

**로깅**  
`langchain_classic.retrievers.multi_query` 로거를 INFO로 올리면  
실제 생성된 질의 목록을 런타임 로그로 확인할 수 있다.

**주의**  
`llm_chain.invoke`의 입력은 문자열이 아니라 `{"question": ...}` 딕셔너리여야 한다.  
(기본 프롬프트의 입력 변수명이 `question`이기 때문)

In [5]:
import logging

from langchain_classic.retrievers.multi_query import MultiQueryRetriever

# 생성된 대체 질의를 로그로 확인
# MultiQueryRetriever 내부 generate_queries()가 INFO로 질의 목록을 남긴다.
logging.basicConfig()
logging.getLogger("langchain_classic.retrievers.multi_query").setLevel(logging.INFO)

# from_llm: 기본 영문 프롬프트 + LineListOutputParser로 llm_chain을 조립한다.
multi_retriever = MultiQueryRetriever.from_llm(
    retriever=base_retriever,
    llm=llm,
    include_original=True,  # 원본 질문도 함께 검색
)

# 생성되는 대체 질의 미리보기
# 검색 전에 llm_chain만 호출해 어떤 패러프레이즈가 나오는지 확인한다.
# 입력 키는 프롬프트 변수명과 동일해야 한다 → {"question": ...}
generated_queries = multi_retriever.llm_chain.invoke({"question": mq_query})
print("=== 생성된 대체 질의 ===")
for i, q in enumerate(generated_queries, 1):
    print(f"{i}. {q}")
print(f"(+ 원본) {mq_query}")

# invoke: 대체 질의(+원본) 각각으로 검색 → unique union
mq_docs_retrieved = multi_retriever.invoke(mq_query)
print(f"\n=== [Multi-Query] 검색 결과 ({len(mq_docs_retrieved)}건, unique) ===")
print(format_docs(mq_docs_retrieved))

# 합쳐진 검색 결과를 문맥으로 최종 답변 생성
mq_answer = answer_chain.invoke(
    {"context": format_docs(mq_docs_retrieved), "question": mq_query}
)
print("\n=== [Multi-Query] 답변 ===")
print(mq_answer)

=== 생성된 대체 질의 ===
1. 질문과 문서의 말투가 다를 때, 검색 범위를 어떻게 확장할 수 있을까?
2. 문서와 질문의 스타일이 다를 경우, 효과적으로 검색 결과를 늘리는 방법은 무엇인가?
3. 질문과 문서의 어조가 상이할 때, 검색 결과를 더 넓히기 위한 전략은 어떤 것이 있을까?
(+ 원본) 질문이랑 문서 말투가 다를 때 검색을 어떻게 넓혀?


INFO:langchain_classic.retrievers.multi_query:Generated queries: ['질문과 문서의 말투가 다를 때, 검색 범위를 어떻게 확장할 수 있을까?', '문서와 질문의 스타일이 다를 경우, 효과적으로 검색 결과를 늘리는 방법은 무엇인가?', '질문과 문서의 어조 차이를 극복하고 검색 결과를 더 다양하게 얻으려면 어떻게 해야 할까?']



=== [Multi-Query] 검색 결과 (2건, unique) ===
[1] (topic=query_expansion) 쿼리 확장은 사용자 질문을 여러 형태로 바꿔 검색 범위를 넓힌다. 동의어·관련어 추가, 질문 재작성, Multi-Query, HyDE 등이 포함된다.

[2] (topic=chunking) 청킹은 긴 문서를 검색·임베딩에 맞는 크기로 나눈다. 너무 작으면 문맥이 끊기고, 너무 크면 노이즈가 섞여 품질이 떨어진다.

=== [Multi-Query] 답변 ===
문맥에 따르면, 질문과 문서의 말투가 다를 때는 쿼리 확장을 통해 검색 범위를 넓힐 수 있습니다. 쿼리 확장은 사용자 질문을 여러 형태로 바꿔 동의어·관련어를 추가하거나 질문을 재작성하는 방법을 포함합니다.


### [A-4] 커스텀 프롬프트로 질의 생성 제어

도메인·언어에 맞게 대체 질의 생성 프롬프트를 바꿀 수 있다.  
출력은 **줄바꿈으로 구분된 질의 목록**이어야 한다 (`LineListOutputParser`).

#### 기술 문서: 질의 재작성 프롬프트 설계

좋은 Multi-Query 프롬프트의 요건:

1. **의도 보존**: 의미가 바뀌면 안 된다. "다른 표현"이지 "다른 질문"이 아니다.
2. **다양성**: 동의어 치환만이 아니라 관점·어휘 레지스터(구어/문어)를 바꾸면 neighborhood 커버가 넓어진다.
3. **파서 친화**: 번호·설명·따옴표 없이 **한 줄 = 한 질의**.
4. **언어 일치**: 코퍼스가 한국어면 대체 질의도 한국어로 생성하는 편이 임베딩 정렬에 유리하다.

기본 프롬프트는 영문 지시라 출력이 혼용될 수 있다.  
도메인 용어(의료, 법률, 내부 제품명)가 있으면 프롬프트에 **허용 용어 목록**을 넣는 것도 효과적이다.

In [6]:
from langchain_core.prompts import PromptTemplate

# 입력 변수명은 반드시 "question"이어야 from_llm 기본 계약과 맞는다.
# 출력 제약: 줄바꿈 구분, 번호/설명 금지 → LineListOutputParser가 그대로 리스트로 파싱한다.
custom_mq_prompt = PromptTemplate(
    input_variables=["question"],
    template=(
        "너는 RAG 검색용 질의 재작성기다.\n"
        "아래 사용자 질문을 의미가 같도록 한국어로 3가지 다른 표현으로 바꿔라.\n"
        "각 줄에 질의 하나만 출력하고, 번호·설명은 넣지 마라.\n\n"
        "원본 질문: {question}"
    ),
)

# prompt= 인자로 기본 영문 프롬프트를 교체한다.
multi_retriever_ko = MultiQueryRetriever.from_llm(
    retriever=base_retriever,
    llm=llm,
    prompt=custom_mq_prompt,
    include_original=True,
)

# 커스텀 프롬프트로 생성된 한국어 대체 질의 확인
ko_queries = multi_retriever_ko.llm_chain.invoke({"question": mq_query})
print("=== [커스텀 프롬프트] 생성된 질의 ===")
for i, q in enumerate(ko_queries, 1):
    print(f"{i}. {q}")

ko_docs = multi_retriever_ko.invoke(mq_query)
print(f"\n검색 문서 수: {len(ko_docs)}")
# topic 라벨로 어떤 주제가 소환됐는지 한눈에 본다.
print("topics:", [d.metadata.get("topic") for d in ko_docs])

=== [커스텀 프롬프트] 생성된 질의 ===
1. 질문과 문서의 말투가 다를 경우 검색 범위를 어떻게 확장할 수 있을까?  
2. 질문과 문서의 어조가 다를 때 검색을 어떻게 더 넓힐 수 있을까?  
3. 질문과 문서의 스타일이 다를 때 검색을 어떻게 다양화할 수 있을까?


INFO:langchain_classic.retrievers.multi_query:Generated queries: ['질문과 문서의 말투가 다를 경우 검색 범위를 어떻게 확장할 수 있을까?  ', '질문과 문서의 어조가 다를 때 검색을 어떻게 더 넓힐 수 있을까?  ', '질문과 문서의 스타일이 다를 때 검색을 어떻게 다양화할 수 있을까?']



검색 문서 수: 2
topics: ['query_expansion', 'chunking']


### [A-5] Naive vs Multi-Query 비교

#### 기술 문서: 검색 품질 비교 방법

소규모 코퍼스에서는 정량 지표(nDCG, Recall@k)보다 **소환된 topic 집합**을 나란히 보는 것이 직관적이다.

| 관찰 | 해석 |
|---|---|
| Multi-Query topic ⊃ Naive topic | 재현율이 늘어난 전형적인 성공 패턴 |
| 집합이 동일 | 이미 단일 질의로 충분했거나, 대체 질의가 너무 유사 |
| 노이즈 topic 증가 | 재작성이 의도를 벗어남 → 프롬프트·temperature 점검 |

실무에서는 Multi-Query 뒤에 **리랭커**를 두면, 넓힌 후보 풀의 정밀도를 다시 끌어올릴 수 있다.  
("넓히고 좁히기": recall↑ → precision↑)

In [7]:
# 표현이 다른 세 가지 질문으로 Naive vs Multi-Query 소환 topic을 비교한다.
test_queries = [
    "질문이랑 문서 말투가 다를 때 검색을 어떻게 넓혀?",
    "키워드랑 의미 검색을 같이 쓰는 방법이 뭐야?",
    "1차로 찾은 문서를 다시 점수 매겨서 순서를 바꾸는 기법은?",
]


def topics(documents: List[Document]) -> List[str]:
    """Document 리스트에서 metadata.topic만 추출한다."""
    return [d.metadata.get("topic", "-") for d in documents]


print(f"{'질문':<38} | {'Naive(k=2)':<28} | {'Multi-Query'}")
print("-" * 110)

for q in test_queries:
    # 왼쪽: 단일 질의 검색 / 오른쪽: 다중 질의 합집합 검색
    naive = topics(base_retriever.invoke(q))
    multi = topics(multi_retriever_ko.invoke(q))
    print(f"{q[:36]:<38} | {str(naive):<28} | {multi}")

질문                                     | Naive(k=2)                   | Multi-Query
--------------------------------------------------------------------------------------------------------------


INFO:langchain_classic.retrievers.multi_query:Generated queries: ['질문과 문서의 말투가 다를 경우 검색 범위를 어떻게 확장할 수 있을까?  ', '질문과 문서의 어조가 다를 때 검색을 어떻게 더 넓힐 수 있을까?  ', '질문과 문서의 스타일이 다를 때 검색을 어떻게 다양화할 수 있을까?']


질문이랑 문서 말투가 다를 때 검색을 어떻게 넓혀?           | ['chunking', 'query_expansion'] | ['query_expansion', 'chunking']


INFO:langchain_classic.retrievers.multi_query:Generated queries: ['키워드와 의미 검색을 함께 사용하는 방법은 무엇인가요?  ', '키워드와 의미 기반 검색을 동시에 활용하는 방법이 궁금해요.  ', '의미 검색과 키워드를 결합해서 사용하는 방법은 어떤 게 있나요?']


키워드랑 의미 검색을 같이 쓰는 방법이 뭐야?              | ['hybrid_search', 'query_expansion'] | ['hybrid_search', 'evaluation', 'query_expansion']


INFO:langchain_classic.retrievers.multi_query:Generated queries: ['1차로 찾은 문서의 순서를 변경하기 위해 점수를 다시 매기는 방법은 무엇인가요?  ', '1차로 검색한 문서에 대해 점수를 재평가하여 순위를 조정하는 기법은?  ', '초기 검색 결과의 문서 점수를 다시 계산하여 순서를 변경하는 기법은 어떤 것이 있나요?']


1차로 찾은 문서를 다시 점수 매겨서 순서를 바꾸는 기법은?      | ['chunking', 'reranker']     | ['reranker', 'chunking', 'hybrid_search']


---
## Part B. Self-Query Retriever

사용자가 "2023년 이후 발행된 RAG 평가 관련 문서만 보여줘"처럼 말하면,  
일반 벡터 검색만으로는 **연도 조건**을 반영하기 어렵다.

`SelfQueryRetriever`는 LLM으로 자연어를 다음 두 가지로 분해한다.
1. **query**: 의미 검색에 쓸 문자열
2. **filter**: 메타데이터 조건 (예: `year >= 2023`, `category == "evaluation"`)

> Self-Query는 메타데이터 필터를 지원하는 벡터스토어가 필요하다.  
> 본 예제에서는 **Chroma**를 사용한다. (FAISS는 Self-Query translator가 없다.)

#### 기술 문서: Self-Query / Structured Query

**핵심 아이디어**  
자연어 질의를 **비구조 의미 검색**과 **구조화 필터**로 분리한다.  
이는 전통 IR의 "키워드 검색 + faceted search(연도·카테고리 패싯)"를 LLM이 자동으로 구성하는 것과 같다.

**내부 파이프라인**

```text
자연어 질문
    │
    ▼
Query Constructor (LLM)
    │  StructuredQuery { query, filter, limit }
    ▼
Translator (예: ChromaTranslator)
    │  벡터스토어 고유 필터 DSL로 변환
    ▼
vectorstore.search(query, filter=..., k=...)
```

**StructuredQuery 구성 요소**
| 필드 | 의미 |
|---|---|
| `query` | 임베딩할 의미 검색 문자열 (필터 문구는 제거된 형태가 이상적) |
| `filter` | Comparison / Operation 트리 (`eq`, `gt`, `and`, `or` 등) |
| `limit` | 반환 개수 상한 (`enable_limit=True`일 때 파싱) |

**Translator의 역할**  
내부 IR 언어(Comparator/Operator)를 Chroma의 `$and`, `$gte` 같은 필터 JSON으로 번역한다.  
벡터스토어마다 DSL이 다르므로 translator가 필수다. FAISS는 공식 translator가 없어 Self-Query에 부적합하다.

**Multi-Query와의 역할 분담**
| | Multi-Query | Self-Query |
|---|---|---|
| 목표 | recall↑ | precision↑ / 조건 강제 |
| 변환 | 질문 → 여러 질문 | 질문 → (검색어 + 필터) |
| 필요 조건 | base retriever만 | 메타데이터 + translator |

### [B-1] 메타데이터가 풍부한 샘플 코퍼스 (Chroma)

#### 기술 문서: 메타데이터 설계와 Chroma

**왜 메타데이터가 중요한가**  
Self-Query의 품질은 LLM 능력뿐 아니라 **인덱스에 실제로 존재하는 필드·값**에 달려 있다.  
스키마에 없는 필드를 물어보면 필터가 실패하거나 빈 결과가 나온다.

**좋은 메타데이터 관례**
- 필터에 쓸 필드는 **정규화된 값**(소문자 enum, 정수 연도)으로 저장한다.
- 자유 서술 문장은 `page_content`에, 기계적 조건은 `metadata`에 둔다.
- 타입을 섞지 않는다. `year`를 문자열 `"2023"`과 정수 `2023`으로 혼용하면 비교 연산이 깨진다.

**Chroma**  
로컬에 쉽게 올릴 수 있는 벡터 DB로, 메타데이터 필터(`$eq`, `$gte`, `$and` 등)를 네이티브 지원한다.  
`collection_name`으로 논리 컬렉션을 구분한다. 같은 이름을 재사용하면 이전 데이터가 남을 수 있으니 실험 시 주의한다.

In [8]:
from langchain_community.vectorstores import Chroma

# Self-Query 데모용 코퍼스: page_content(의미) + 구조화 metadata(필터).
# year는 반드시 int, level/category/author는 정규화된 문자열로 통일한다.
sq_docs = [
    Document(
        page_content=(
            "하이브리드 검색은 BM25와 벡터 검색을 결합해 키워드 매칭과 의미 유사성을 "
            "동시에 활용한다. RRF 융합이 실무에서 자주 쓰인다."
        ),
        metadata={
            "topic": "hybrid_search",
            "category": "retrieval",
            "year": 2022,
            "level": "intermediate",
            "author": "kim",
        },
    ),
    Document(
        page_content=(
            "리랭커는 bi-encoder로 넓은 후보를 가져온 뒤 cross-encoder로 재정렬한다. "
            "상위 k의 정밀도를 크게 올릴 수 있다."
        ),
        metadata={
            "topic": "reranker",
            "category": "retrieval",
            "year": 2023,
            "level": "advanced",
            "author": "lee",
        },
    ),
    Document(
        page_content=(
            "Multi-Query는 질문을 여러 관점으로 재작성해 검색 재현율을 높인다. "
            "HyDE는 가상 답변 문서를 만들어 그 임베딩으로 검색한다."
        ),
        metadata={
            "topic": "query_expansion",
            "category": "retrieval",
            "year": 2022,
            "level": "intermediate",
            "author": "park",
        },
    ),
    Document(
        page_content=(
            "RecursiveCharacterTextSplitter는 구분자 우선순위로 긴 문서를 청크로 나눈다. "
            "청크 크기와 overlap 선택이 검색 품질에 큰 영향을 준다."
        ),
        metadata={
            "topic": "chunking",
            "category": "preprocessing",
            "year": 2021,
            "level": "beginner",
            "author": "choi",
        },
    ),
    Document(
        page_content=(
            "text-embedding-3-small은 비용 대비 성능이 좋은 OpenAI 임베딩이다. "
            "다국어·짧은 질의에도 비교적 안정적인 검색 성능을 보인다."
        ),
        metadata={
            "topic": "embedding",
            "category": "embedding",
            "year": 2024,
            "level": "beginner",
            "author": "kim",
        },
    ),
    Document(
        page_content=(
            "RAGAS는 faithfulness, answer relevancy, context precision, context recall로 "
            "RAG 시스템의 검색·생성 품질을 정량 평가하는 프레임워크이다."
        ),
        metadata={
            "topic": "evaluation",
            "category": "evaluation",
            "year": 2024,
            "level": "advanced",
            "author": "jung",
        },
    ),
    Document(
        page_content=(
            "LLM-as-a-Judge는 대형 언어모델로 답변의 관련성·근거 일치 여부를 채점한다. "
            "사람 평가 비용을 줄이면서도 비교적 일관된 점수를 얻을 수 있다."
        ),
        metadata={
            "topic": "evaluation",
            "category": "evaluation",
            "year": 2023,
            "level": "advanced",
            "author": "jung",
        },
    ),
    Document(
        page_content=(
            "프롬프트 캐싱과 배치 임베딩은 RAG 파이프라인의 지연·비용을 줄이는 운영 기법이다. "
            "트래픽이 많은 서비스에서 특히 중요하다."
        ),
        metadata={
            "topic": "ops",
            "category": "ops",
            "year": 2024,
            "level": "intermediate",
            "author": "lee",
        },
    ),
]

# Chroma 컬렉션 생성. embedding은 FAISS와 동일한 모델을 사용해 비교 조건을 맞춘다.
# collection_name이 같으면 이전 실행 데이터가 누적될 수 있다.
sq_vectorstore = Chroma.from_documents(
    documents=sq_docs,
    embedding=embeddings,
    collection_name="self_query_demo",
)

print(f"Self-Query용 문서 수: {len(sq_docs)}")
print("메타데이터 필드 예:", sq_docs[0].metadata)

Self-Query용 문서 수: 8
메타데이터 필드 예: {'topic': 'hybrid_search', 'category': 'retrieval', 'year': 2022, 'level': 'intermediate', 'author': 'kim'}


### [B-2] `AttributeInfo`로 메타데이터 스키마 정의

LLM이 어떤 필드로 필터를 만들 수 있는지 **설명**을 제공해야 한다.  
`description`이 구체적일수록 필터 생성이 안정적이다.

> **`structured_query_translator`를 명시하는 이유**  
> 생략하면 LangChain이 벡터스토어 종류를 자동 감지하면서 `langchain_community`의 모든 벡터스토어를 임포트한다.  
> 이 과정에서 버전에 따라 제거된 클래스가 있으면 `ImportError: cannot import name 'DatabricksVectorSearch' ...` 같은 오류가 난다.  
> Chroma를 쓸 때는 `ChromaTranslator()`를 직접 넘겨 자동 감지 경로를 건너뛴다.

#### 기술 문서: AttributeInfo와 Query Constructor

**`AttributeInfo`** 는 LLM에게 보여주는 **필터 스키마 카드**다.

| 필드 | 역할 |
|---|---|
| `name` | 실제 metadata 키와 일치해야 한다 |
| `type` | `string`, `integer`, `float`, `boolean` 등 — 비교 연산자 선택에 영향 |
| `description` | 허용 값·의미를 자연어로 설명. few-shot처럼 동작한다 |

**`document_contents`**  
코퍼스 전체 요약이다. LLM이 "이 DB에는 무엇이 있는지"를 파악해 `query` 문자열을 다듬는 데 쓴다.

**`enable_limit=True`**  
"상위 2개만" 같은 자연어를 `StructuredQuery.limit`으로 파싱한다.  
끄면 limit은 항상 `None`이고 `search_kwargs["k"]`만 적용된다.

**ChromaTranslator 허용 연산**
- Comparator: `eq`, `ne`, `gt`, `gte`, `lt`, `lte`
- Operator: `and`, `or`
- (참고) `in` / `nin` 등은 스토어·translator 조합에 따라 다를 수 있다.

In [9]:
from langchain_classic.chains.query_constructor.schema import AttributeInfo
from langchain_classic.retrievers.self_query.base import SelfQueryRetriever
from langchain_community.query_constructors.chroma import ChromaTranslator

# LLM이 필터를 만들 때 참고하는 스키마.
# description에 허용 값을 명시하면 잘못된 attribute/value 생성이 줄어든다.
metadata_field_info = [
    AttributeInfo(
        name="topic",
        description=(
            "문서 주제. 가능한 값: hybrid_search, reranker, query_expansion, "
            "chunking, embedding, evaluation, ops"
        ),
        type="string",
    ),
    AttributeInfo(
        name="category",
        description=(
            "대분류. 가능한 값: retrieval, preprocessing, embedding, evaluation, ops"
        ),
        type="string",
    ),
    AttributeInfo(
        name="year",
        description="문서가 다루는(또는 발행된) 연도. 정수",
        type="integer",
    ),
    AttributeInfo(
        name="level",
        description="난이도. 가능한 값: beginner, intermediate, advanced",
        type="string",
    ),
    AttributeInfo(
        name="author",
        description="작성자 성. 가능한 값: kim, lee, park, choi, jung",
        type="string",
    ),
]

# 코퍼스 요약: Query Constructor가 의미 검색어(query)를 다듬을 때 참고한다.
document_contents = (
    "RAG(검색증강생성) 관련 기술 노트. "
    "검색, 전처리, 임베딩, 평가, 운영에 대한 설명 문서."
)

self_query_retriever = SelfQueryRetriever.from_llm(
    llm=llm,
    vectorstore=sq_vectorstore,
    document_contents=document_contents,
    metadata_field_info=metadata_field_info,
    # translator를 생략하면 벡터스토어 종류를 자동 감지하는데,
    # 이때 langchain_community의 모든 벡터스토어를 임포트하다가
    # 버전에 따라 ImportError가 날 수 있어 명시적으로 지정한다.
    structured_query_translator=ChromaTranslator(),
    enable_limit=True,  # "상위 2개만" 같은 자연어 limit도 파싱
    search_kwargs={"k": 4},  # limit이 없을 때 기본 반환 개수
)

print("SelfQueryRetriever 생성 완료")
print(f"타입: {type(self_query_retriever)}")

SelfQueryRetriever 생성 완료
타입: <class 'langchain_classic.retrievers.self_query.base.SelfQueryRetriever'>


### [B-3] 구조화 쿼리 분해 확인

`query_constructor`로 LLM이 만든 structured query(검색어 + 필터)를 직접 볼 수 있다.

#### 기술 문서: StructuredQuery 디버깅

Self-Query 오류의 대부분은 "검색이 이상하다"가 아니라 **필터 분해가 틀린 경우**다.  
검색 전에 `query_constructor.invoke({"query": ...})`로 중간 산출물을 확인하는 습관이 중요하다.

**읽는 법**
- `query`: 임베딩에 들어갈 문자열. 필터 문구("2023년 이후")가 여기 남아 있으면 노이즈가 될 수 있다.
- `filter`: `Comparison(attribute, comparator, value)` 또는 `Operation(and/or, [...])`
- `limit`: 정수 또는 `None`

**자주 보는 패턴**
| 사용자 말 | 기대 분해 |
|---|---|
| "2023년 이후 평가 문서" | `query≈평가`, `year > 2023` 또는 `>=`, `category/topic` 조건 |
| "리랭커가 뭐야?" | `query≈리랭커`, `filter=None` (순수 의미 검색) |
| "jung 글 2개만" | `author=jung`, `limit=2` |

In [10]:
def show_structured_query(question: str) -> None:
    """자연어 질문을 StructuredQuery로 분해해 출력한다.

    실제 벡터 검색 전에 LLM이 어떤 query/filter/limit을 만들었는지
    검증할 때 사용한다. 입력을 반드시 {"query": ...} 형태로 넘긴다.
    """
    structured = self_query_retriever.query_constructor.invoke({"query": question})
    print(f"질문: {question}")
    print(f"  query : {structured.query!r}")
    print(f"  filter: {structured.filter}")
    print(f"  limit : {structured.limit}")
    print()


# 필터가 있는 질문 / 없는 질문을 섞어 분해 패턴을 관찰한다.
demo_questions = [
    "2023년 이후 나온 평가(evaluation) 관련 문서를 찾아줘",
    "beginner 난이도의 전처리(preprocessing) 문서만",
    "kim이 쓴 임베딩 관련 글",
    "리랭커가 뭐야?",  # 필터 없이 의미 검색만
]

for q in demo_questions:
    show_structured_query(q)

질문: 2023년 이후 나온 평가(evaluation) 관련 문서를 찾아줘
  query : '평가'
  filter: comparator=<Comparator.GT: 'gt'> attribute='year' value=2023
  limit : None

질문: beginner 난이도의 전처리(preprocessing) 문서만
  query : '전처리'
  filter: comparator=<Comparator.EQ: 'eq'> attribute='level' value='beginner'
  limit : None

질문: kim이 쓴 임베딩 관련 글
  query : '임베딩'
  filter: comparator=<Comparator.EQ: 'eq'> attribute='author' value='kim'
  limit : None

질문: 리랭커가 뭐야?
  query : '리랭커'
  filter: None
  limit : None



### [B-4] Self-Query 검색 & RAG 답변

#### 기술 문서: 필터 유무에 따른 결과 차이

같은 자연어라도:

- **일반 `similarity_search`**: 의미 유사도만 본다. 연도·카테고리 조건은 "문장에 그 단어가 있으면" 간접적으로만 반영된다.
- **Self-Query**: 조건을 메타데이터 필터로 **강제**한다. 의미적으로 가깝더라도 필터에 안 맞으면 탈락한다.

따라서 Self-Query는 "관련성"과 "적격성(eligibility)"을 분리한다.  
RAG 답변은 필터를 통과한 문서만 문맥으로 쓰므로, 할루시네이션성 오래된/잘못된 카테고리 근거가 줄어든다.

**주의**: LLM이 필터를 너무 약하게 만들면(예: year만 적용하고 category 누락)  
일반 검색과 비슷한 잡음이 다시 섞일 수 있다. 이때는 `AttributeInfo` description을 더 구체화하거나 few-shot을 `chain_kwargs`로 보강한다.

In [11]:
sq_query = "2023년 이후 나온 RAG 평가 관련 문서를 설명해줘"

# 비교용: 필터 없는 일반 검색
# 의미 유사도만 사용하므로 year/category 조건이 강제되지 않는다.
plain_docs = sq_vectorstore.similarity_search(sq_query, k=3)
print("=== [일반 벡터 검색] ===")
print(format_docs(plain_docs))

# Self-Query: 연도/카테고리 필터 자동 적용
# 내부적으로 query_constructor → translator → filtered search 순으로 실행된다.
sq_retrieved = self_query_retriever.invoke(sq_query)
print("\n=== [Self-Query] ===")
print(format_docs(sq_retrieved))

# 필터를 통과한 문서만 근거로 답변 생성
sq_answer = answer_chain.invoke(
    {"context": format_docs(sq_retrieved), "question": sq_query}
)
print("\n=== [Self-Query] 답변 ===")
print(sq_answer)

=== [일반 벡터 검색] ===
[1] (level=advanced, year=2024, author=jung, topic=evaluation, category=evaluation) RAGAS는 faithfulness, answer relevancy, context precision, context recall로 RAG 시스템의 검색·생성 품질을 정량 평가하는 프레임워크이다.

[2] (level=intermediate, category=ops, author=lee, topic=ops, year=2024) 프롬프트 캐싱과 배치 임베딩은 RAG 파이프라인의 지연·비용을 줄이는 운영 기법이다. 트래픽이 많은 서비스에서 특히 중요하다.

[3] (author=lee, topic=reranker, category=retrieval, year=2023, level=advanced) 리랭커는 bi-encoder로 넓은 후보를 가져온 뒤 cross-encoder로 재정렬한다. 상위 k의 정밀도를 크게 올릴 수 있다.

=== [Self-Query] ===
[1] (topic=evaluation, category=evaluation, year=2024, author=jung, level=advanced) RAGAS는 faithfulness, answer relevancy, context precision, context recall로 RAG 시스템의 검색·생성 품질을 정량 평가하는 프레임워크이다.

[2] (category=ops, author=lee, level=intermediate, topic=ops, year=2024) 프롬프트 캐싱과 배치 임베딩은 RAG 파이프라인의 지연·비용을 줄이는 운영 기법이다. 트래픽이 많은 서비스에서 특히 중요하다.

[3] (author=jung, level=advanced, year=2023, category=evaluation, topic=evaluation) LLM-as-a-Judge는 대형 언어모델로 답변의 관련성·근거 일치 여

### [B-5] 다양한 필터 질의 실험

#### 기술 문서: 복합 필터와 엣지 케이스

Self-Query가 다루는 대표적인 질의 유형:

| 유형 | 예 | 기대 동작 |
|---|---|---|
| AND 복합 | "advanced이면서 evaluation" | `level=eq` ∧ `category=eq` |
| 범위 비교 | "2022년 이전" | `year < 2022` (또는 `<=`) |
| limit | "상위 2개만" | `limit=2` |
| 순수 의미 | "청킹 방법 알려줘" | `filter=None`, query만 사용 |

**빈 결과(0건)** 도 버그가 아닐 수 있다.  
예: "2022년 이전 retrieval"인데 코퍼스에 `category=retrieval`이면서 `year<2022`인 문서가 없으면 정상적으로 빈 집합이다.  
이때는 스키마·데이터·비교 연산(`lt` vs `lte`)을 함께 점검한다.

**`query`가 공백 `' '`인 경우**  
필터만으로 충분하다고 LLM이 판단하면 의미 검색어를 비우다시피 할 수 있다.  
스토어에 따라 빈 쿼리 동작이 다르므로, 필요하면 `use_original_query=True`로 원본 질문을 검색어로 강제할 수 있다.

In [12]:
# AND / 범위 / limit / 무필터 등 다양한 자연어 패턴을 실험한다.
filter_queries = [
    "advanced 난이도이면서 category가 evaluation인 문서",
    "2022년 이전의 retrieval 카테고리 문서",
    "author가 jung인 문서 상위 2개만",
    "청킹이랑 문서 나누는 방법 알려줘",  # 필터 없을 수 있음
]

for q in filter_queries:
    print("=" * 70)
    # 1) 분해 결과 확인 (디버깅)
    show_structured_query(q)
    # 2) 실제 검색 실행
    docs = self_query_retriever.invoke(q)
    print(f"검색 결과 {len(docs)}건:")
    for d in docs:
        # metadata만 요약 출력해 필터 적합성(eligibility)을 빠르게 검증한다.
        print(
            f"  - year={d.metadata.get('year')}, "
            f"category={d.metadata.get('category')}, "
            f"level={d.metadata.get('level')}, "
            f"author={d.metadata.get('author')}, "
            f"topic={d.metadata.get('topic')}"
        )
    print()

질문: advanced 난이도이면서 category가 evaluation인 문서
  query : ' '
  filter: operator=<Operator.AND: 'and'> arguments=[Comparison(comparator=<Comparator.EQ: 'eq'>, attribute='level', value='advanced'), Comparison(comparator=<Comparator.EQ: 'eq'>, attribute='category', value='evaluation')]
  limit : None

검색 결과 2건:
  - year=2024, category=evaluation, level=advanced, author=jung, topic=evaluation
  - year=2023, category=evaluation, level=advanced, author=jung, topic=evaluation

질문: 2022년 이전의 retrieval 카테고리 문서
  query : ' '
  filter: operator=<Operator.AND: 'and'> arguments=[Comparison(comparator=<Comparator.EQ: 'eq'>, attribute='category', value='retrieval'), Comparison(comparator=<Comparator.LT: 'lt'>, attribute='year', value=2022)]
  limit : None

검색 결과 0건:

질문: author가 jung인 문서 상위 2개만
  query : ' '
  filter: comparator=<Comparator.EQ: 'eq'> attribute='author' value='jung'
  limit : 2

검색 결과 2건:
  - year=2024, category=evaluation, level=advanced, author=jung, topic=evaluation
  - year=2023, ca

### [B-6] (참고) 필터만 수동으로 적용하기

Self-Query 없이 Chroma 메타데이터 필터를 직접 넘기는 방식과 비교한다.

#### 기술 문서: 수동 필터 vs Self-Query

Chroma(및 많은 벡터 DB)는 검색 API에 필터 dict를 직접 받는다.

```python
filter={
    "$and": [
        {"year": {"$gte": 2023}},
        {"category": "evaluation"},
    ]
}
```

| 방식 | 장점 | 단점 |
|---|---|---|
| **수동 필터** | 결정적·디버깅 쉬움, LLM 비용 없음 | UI/API에서 조건을 직접 구성해야 함 |
| **Self-Query** | 자연어로 조건 표현, UX 좋음 | LLM 파싱 오류·추가 지연·스키마 의존 |

실무 패턴:
- 챗봇·자연어 검색창 → Self-Query
- 관리 콘솔·고정 필터 UI → 수동 필터
- 하이브리드: UI 패싯은 수동, 자유 텍스트만 Self-Query로 보조

In [13]:
# Chroma 수동 필터 예: year >= 2023 AND category == "evaluation"
# Self-Query translator가 만들어 내는 DSL과 동일한 형태다.
# LLM 없이 조건을 코드로 고정할 때 이 방식을 쓴다.
manual_filtered = sq_vectorstore.similarity_search(
    "RAG 평가",  # 의미 검색어
    k=4,
    filter={
        "$and": [
            {"year": {"$gte": 2023}},
            {"category": "evaluation"},
        ]
    },
)

print("=== 수동 메타데이터 필터 ===")
print(format_docs(manual_filtered))
print(
    "\nSelf-Query는 위와 같은 필터를 자연어에서 자동으로 만들어 준다."
)

=== 수동 메타데이터 필터 ===
[1] (year=2024, author=jung, topic=evaluation, category=evaluation, level=advanced) RAGAS는 faithfulness, answer relevancy, context precision, context recall로 RAG 시스템의 검색·생성 품질을 정량 평가하는 프레임워크이다.

[2] (year=2023, category=evaluation, level=advanced, topic=evaluation, author=jung) LLM-as-a-Judge는 대형 언어모델로 답변의 관련성·근거 일치 여부를 채점한다. 사람 평가 비용을 줄이면서도 비교적 일관된 점수를 얻을 수 있다.

Self-Query는 위와 같은 필터를 자연어에서 자동으로 만들어 준다.


---
### [정리] Multi-Query vs Self-Query vs HyDE

| 기법 | 무엇을 바꾸는가 | 강점 | 비용/주의 |
|---|---|---|---|
| **Multi-Query** | 질문 → 여러 대체 질문 | 재현율↑, 표현 다양성 대응 | 질의 수만큼 검색 호출 |
| **Self-Query** | 질문 → (검색어 + 메타 필터) | 연도·카테고리 등 조건부 검색 | 메타데이터 스키마·벡터스토어 지원 필요 |
| **HyDE** | 질문 → 가상 문서 임베딩 | 질문-문서 문체 간극 완화 | 가상 문서는 최종 근거로 쓰지 않음 |

**구현 포인트 (LangChain v1)**
- Multi-Query: `from langchain_classic.retrievers.multi_query import MultiQueryRetriever`
- Self-Query: `from langchain_classic.retrievers.self_query.base import SelfQueryRetriever`
- 스키마: `from langchain_classic.chains.query_constructor.schema import AttributeInfo`
- Self-Query 벡터스토어: Chroma / Pinecone / PGVector 등 (FAISS는 translator 미지원)
- Chroma 사용 시: `structured_query_translator=ChromaTranslator()` 명시 권장

**언제 무엇을**
- 표현이 흔들리거나 관련 문서를 놓치기 싫다 → **Multi-Query**
- "2024년 자료만", "advanced만"처럼 조건이 있다 → **Self-Query**
- 질문이 너무 짧아 문서와 표현이 어긋난다 → **HyDE** (이전 노트북)

#### 기술 문서: 실무 조합 패턴

```text
사용자 질문
    │
    ├─ 메타데이터 조건 포함? ──yes──► Self-Query (필터 강제)
    │
    ├─ 질문이 짧거나 문체 불일치? ──yes──► HyDE 또는 Multi-Query
    │
    └─ 후보가 충분해진 뒤 ──► Reranker로 precision 회복
```

세 기법은 배타적이지 않다. 예를 들어 Self-Query로 연도·카테고리를 걸고,  
남은 의미 검색어에 Multi-Query를 적용하는 2단 파이프라인도 가능하다.  
다만 LLM 호출이 중첩되면 지연·비용이 커지므로, 라우터(규칙 또는 작은 분류 모델)로 필요한 변환만 켜는 것이 실무적이다.